In [ ]:
import pennylane as qml

In [ ]:
import jax

In [ ]:
# pip install jaxopt


In [ ]:
from jax import numpy as jnp
import jaxopt

In [ ]:
jax.config.update("jax_platform_name", "cpu")

In [ ]:
n_wires = 5
data = jnp.sin(jnp.mgrid[-2:2:0.2].reshape(n_wires, -1)) ** 3
targets = jnp.array([-0.2, 0.4, 0.35, 0.2])

In [ ]:
print(data)

[[-7.5182688e-01 -9.2357683e-01 -9.9872130e-01 -9.5698124e-01]
 [-8.0965924e-01 -5.9582293e-01 -3.6915100e-01 -1.8001968e-01]
 [-5.9053835e-02 -7.8413300e-03  1.0842022e-19  7.8414418e-03]
 [ 5.9054226e-02  1.8002041e-01  3.6915201e-01  5.9582406e-01]
 [ 8.0966020e-01  9.5698160e-01  9.9872130e-01  9.2357635e-01]]


In [ ]:
dev = qml.device("default.qubit", wires=n_wires)

In [ ]:
@qml.qnode(dev)
def circuit(data, weights):


    # data embedding
    for i in range(n_wires):
        qml.RY(data[i], wires=i)
        print(data)


    for i in range(n_wires):
        qml.RX(weights[i, 0], wires=i)
        print(data)
        qml.RY(weights[i, 1], wires=i)
        qml.RX(weights[i, 2], wires=i)
        qml.CNOT(wires=[i, (i + 1) % n_wires])

    # we use a sum of local Z's as an observable since a
    # local Z would only be affected by params on that qubit.
    return qml.expval(qml.sum(*[qml.PauliZ(i) for i in range(n_wires)]))



In [ ]:
def my_model(data, weights, bias):
    return circuit(data, weights) + bias

In [ ]:
@jax.jit
def loss_fn(params, data, targets):
    predictions = my_model(data, params["weights"], params["bias"])
    loss = jnp.sum((targets - predictions) ** 2 / len(data))
    return loss

In [ ]:
weights = jnp.ones([n_wires, 3])
bias = jnp.array(0.)
params = {"weights": weights, "bias": bias}

In [ ]:
print(loss_fn(params, data, targets))

print(jax.grad(loss_fn)(params, data, targets))

Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
Traced<ShapedArray(float32[5,4])>with<DynamicJaxprTrace(level=1/0)>
0.29232618
{'bias': Array(-0.754321, dtype=float32, weak_type=True), 'weights': Array([[-1.9507733e-01,  5.2854650e-02, -4.8925212e-01],
       [-1.9968867e-02, -5.3287148e-02,  9.2290469e-02],
       [-2.7175695e-03, -9.6455216e-05, -4.7958046e-03],
       [-6.3544422e-02,  3.6111072e-02, -2.0519713e-01],
       [-9.0

In [ ]:
def loss_and_grad(params, data, targets, print_training, i):
    loss_val, grad_val = jax.value_and_grad(loss_fn)(params, data, targets)

    def print_fn():
        jax.debug.print("Step: {i}  Loss: {loss_val}", i=i, loss_val=loss_val)

    # if print_training=True, print the loss every 5 steps
    jax.lax.cond((jnp.mod(i, 5) == 0) & print_training, print_fn, lambda: None)

    return loss_val, grad_val

In [ ]:
opt = jaxopt.GradientDescent(loss_and_grad, stepsize=0.3, value_and_grad=True)
opt_state = opt.init_state(params)

for i in range(100):
    params, opt_state = opt.update(params, opt_state, data, targets, True, i)

Step: 0  Loss: 0.2923261821269989
Step: 5  Loss: 0.08928847312927246
Step: 10  Loss: 0.07159452140331268
Step: 15  Loss: 0.0573359839618206
Step: 20  Loss: 0.047165658324956894
Step: 25  Loss: 0.039545513689517975
Step: 30  Loss: 0.033213984221220016
Step: 35  Loss: 0.027763623744249344
Step: 40  Loss: 0.02355431765317917
Step: 45  Loss: 0.02116141840815544
Step: 50  Loss: 0.020479023456573486
Step: 55  Loss: 0.020495953038334846
Step: 60  Loss: 0.020188236609101295
Step: 65  Loss: 0.019282221794128418
Step: 70  Loss: 0.018023068085312843
Step: 75  Loss: 0.016644064337015152
Step: 80  Loss: 0.015254518948495388
Step: 85  Loss: 0.013919454999268055
Step: 90  Loss: 0.012653568759560585
Step: 95  Loss: 0.011443670839071274


In [ ]:
@jax.jit
def optimization_jit(params, data, targets, print_training=False):
    opt = jaxopt.GradientDescent(loss_and_grad, stepsize=0.3, value_and_grad=True)
    opt_state = opt.init_state(params)

    def update(i, args):
        params, opt_state = opt.update(*args, i)
        return (params, opt_state, *args[2:])

    args = (params, opt_state, data, targets, print_training)
    (params, opt_state, _, _, _) = jax.lax.fori_loop(0, 100, update, args)

    return params

In [ ]:
params = {"weights": weights, "bias": bias}
optimization_jit(params, data, targets, print_training=True)

Step: 0  Loss: 0.2923261821269989
Step: 5  Loss: 0.08928847312927246
Step: 10  Loss: 0.07159452140331268
Step: 15  Loss: 0.0573359839618206
Step: 20  Loss: 0.047165658324956894
Step: 25  Loss: 0.039545513689517975
Step: 30  Loss: 0.033213984221220016
Step: 35  Loss: 0.027763623744249344
Step: 40  Loss: 0.02355431765317917
Step: 45  Loss: 0.02116141840815544
Step: 50  Loss: 0.020479023456573486
Step: 55  Loss: 0.020495953038334846
Step: 60  Loss: 0.020188236609101295
Step: 65  Loss: 0.019282221794128418
Step: 70  Loss: 0.018023068085312843
Step: 75  Loss: 0.016644064337015152
Step: 80  Loss: 0.015254518948495388
Step: 85  Loss: 0.013919454999268055
Step: 90  Loss: 0.012653568759560585
Step: 95  Loss: 0.011443670839071274


{'bias': Array(-0.90738827, dtype=float32, weak_type=True),
 'weights': Array([[ 1.5734389 ,  1.4267787 ,  0.5062411 ],
        [ 0.17005834,  0.83468944,  1.9624869 ],
        [ 1.4379824 ,  1.1278558 ,  2.2343845 ],
        [-0.2590661 ,  0.5319227 ,  1.3994671 ],
        [ 1.2112952 ,  1.6135516 ,  3.1225502 ]], dtype=float32)}

In [ ]:
from timeit import repeat

def optimization(params, data, targets):
    opt = jaxopt.GradientDescent(loss_and_grad, stepsize=0.3, value_and_grad=True)
    opt_state = opt.init_state(params)

    for i in range(100):
        params, opt_state = opt.update(params, opt_state, data, targets, False, i)

    return params

reps = 5
num = 2

times = repeat("optimization(params, data, targets)", globals=globals(), number=num, repeat=reps)
result = min(times) / num

print(f"Jitting just the cost (best of {reps}): {result} sec per loop")

times = repeat("optimization_jit(params, data, targets)", globals=globals(), number=num, repeat=reps)
result = min(times) / num

print(f"Jitting the entire optimization (best of {reps}): {result} sec per loop")

Jitting just the cost (best of 5): 1.3670296049995159 sec per loop
Jitting the entire optimization (best of 5): 0.004701302999819745 sec per loop


expval(Z(1))
<default.qubit device (wires=5) at 0x7cdbc2117f40>


Expectation value of Pauli-Z on qubit 1: 0.0
